# Final Model Evaluation

## 1. Objective

The tuned class-weighted random forest achieved the strongest validation F1 score and was selected as the final model.

This notebook will:

- Combine the training and validation datasets.
- Retrain the selected model using the combined development data.
- Evaluate the model once on the untouched test set.
- Report accuracy, precision, recall, F1, and AUC-ROC.
- Examine the confusion matrix and ROC curve.
- Summarize the final model's performance.

The test set was not used during model selection or hyperparameter tuning.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

## 2. Load the processed datasets

In [ ]:
project_root = (
    Path.cwd().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd()
)

processed_dir = project_root / 'data' / 'processed'

In [ ]:
X_train = pd.read_csv(processed_dir / 'X_train.csv')
X_val = pd.read_csv(processed_dir / 'X_val.csv')
X_test = pd.read_csv(processed_dir / 'X_test.csv')

y_train = pd.read_csv(processed_dir / 'y_train.csv')['Exited']
y_val = pd.read_csv(processed_dir / 'y_val.csv')['Exited']
y_test = pd.read_csv(processed_dir / 'y_test.csv')['Exited']

In [ ]:
print(f'X_train: {X_train.shape}')
print(f'X_val: {X_val.shape}')
print(f'X_test: {X_test.shape}')

print(f'y_train: {y_train.shape}')
print(f'y_val: {y_val.shape}')
print(f'y_test: {y_test.shape}')

In [ ]:
# Integrity checks

assert X_train.columns.equals(X_val.columns)
assert X_train.columns.equals(X_test.columns)

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

assert X_train.isna().sum().sum() == 0
assert X_val.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

print('All datasets verified.')

## 3. Create the development dataset

The training and validation sets are combined after model selection. This gives the final model more observations for training while preserving the test set for one unbiased final evaluation.

In [ ]:
X_development = pd.concat(
    [X_train, X_val],
    ignore_index=True
)

y_development = pd.concat(
    [y_train, y_val],
    ignore_index=True
)

print(f'X_development: {X_development.shape}')
print(f'y_development: {y_development.shape}')

print('\nDevelopment target proportions:')
print(y_development.value_counts(normalize=True).round(4))

## 4. Train the selected model

The selected class-weighted random forest is recreated using the hyperparameters chosen in the tuning notebook. It is then trained on the combined training and validation data.

In [ ]:
final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

final_model.fit(
    X_development,
    y_development
)

print('Final model trained.')

## 5. Final Test Evaluation

The trained model is now evaluated once on the untouched test set. No additional model selection or hyperparameter tuning will be performed using these results.

In [ ]:
test_predictions = final_model.predict(X_test)
test_probabilities = final_model.predict_proba(X_test)[:, 1]

In [ ]:
test_results = {
    'Accuracy': accuracy_score(
        y_test,
        test_predictions
    ),
    'Precision': precision_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    'Recall': recall_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    'F1': f1_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    'AUC-ROC': roc_auc_score(
        y_test,
        test_probabilities
    )
}

test_results_df = pd.DataFrame(
    [test_results],
    index=['Final Random Forest']
)

test_results_df.round(4)